# Chapter 8: RAG Generation — The Answer Layer
## Topic 1: Prompt Construction from Retrieved Chunks — Context Window Budgeting

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapter 7 ends with a ranked, filtered, deduplicated list of chunks. Those chunks are still just data sitting in memory — this topic is where they actually become a prompt string sent to a language model.
- The core problem: a language model has a fixed context window. Every token spent on retrieved context is a token not available for the system prompt, conversation history, and the model's own output. Prompt construction is the discipline of deliberately deciding what goes into that limited space and in what order.
- Why this can't just be "concatenate the chunks together": naive concatenation ignores three things that materially affect answer quality — how much budget is actually available after everything else is accounted for, whether chunk order affects how well the model uses the content (it does — this is the "lost in the middle" effect), and whether the model can tell where one chunk ends and another begins, and which source each piece of content came from (needed for proper citation later).


### 2. Internal Working — Step by Step

1. **Budget calculation:** take the total context window size, subtract the system prompt's token count, subtract any tool-schema tokens, subtract conversation history tokens (for multi-turn cases), and subtract a reserved allocation for the model's own output. What remains is the actual budget available for retrieved chunks.
2. **Token counting:** actual token count — not character count or word count — is what matters, since context windows are measured in tokens. Using the real tokenizer (or a carefully calibrated approximation) to estimate this is important; simple character-based heuristics can silently overflow on dense or non-English text.
3. **Chunk selection under budget:** walk the ranked chunk list (already ordered by the retrieval and reranking pipeline) and greedily include chunks until the token budget is exhausted. Never truncate a chunk mid-sentence to make it fit — drop the whole chunk instead.
4. **Formatting each chunk with source attribution:** each chunk should be wrapped with an explicit marker identifying where it came from. This is the raw material any later citation mechanism depends on, and it must be present at construction time, not added afterward as an afterthought.
5. **Ordering for the "lost in the middle" effect:** the most relevant chunk (by the final ranking) should sit at the beginning or end of the context block, not buried in the middle, since language models empirically attend less reliably to information placed in the middle of a long context, even when that information technically fits within the window.
6. **Assembling the final prompt:** the system prompt (rules, task definition), the formatted context block (the chunks), and the user's actual query are combined into the final structure sent to the model.


### 3. How This Is Implemented in This Project

- The project's existing agent pattern already sends a system prompt and a list of role/content messages to the model — this topic extends that exact pattern by inserting the retrieved chunks into the message list before the actual query, rather than relying purely on the model's own general knowledge.
- The retrieval step already returns a ranked list of scored documents — this topic's job is turning that list into a well-formatted context block, respecting the available budget, before it ever reaches the actual model call.
- The specific model in use has a defined context window size and per-token cost — both numbers matter directly: the window size drives the budgeting math, and the cost drives the discussion of what happens if budgets and reservations are set carelessly.
- A rough character-based token estimate (roughly a fixed number of characters per token) is a reasonable starting approximation for English text, but is systematically less reliable for Hinglish or other mixed-script content — a real tokenizer call should always be preferred when precision matters, especially near the edge of the budget.
- The core logic: greedily walk the ranked chunks, formatting each with an explicit source tag, tracking cumulative token usage, and stopping (dropping the rest, never truncating) once the budget would be exceeded. The final prompt combines the available budget calculation, the resulting context block, and the user's actual query into the message sent to the model.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Silent truncation is worse than an outright error:** if budgeting logic is flawed and a chunk gets cut off mid-sentence, the model may confidently answer based on incomplete information with no signal that anything went wrong. The rule should always be to drop whole chunks rather than truncate them, and to log whenever a chunk gets dropped due to budget constraints.
- **Token estimation drift:** character-based approximations are systematically less accurate for Hinglish and other code-mixed text. For a corpus with substantial non-English content, relying on a rough estimate near the budget edge risks either wasting available headroom or genuinely overflowing the context window. Periodically calibrating the estimate against the real tokenizer's actual output is worth building into the pipeline.
- **Forgetting to reserve output budget:** if no tokens are reserved for the model's own response, a genuinely long, correct answer can get cut off mid-generation — this is a distinct failure mode from context overflow, but equally damaging to answer quality.
- **Scaling and latency:** token counting and chunk selection are cheap, fast operations — the real latency cost in this stage is negligible compared to retrieval and generation themselves. This step scales easily even at high request volume.
- **Cost:** every token included in the context block is a token billed as model input on every single request — a system prompt or context block that's larger than necessary is a direct, ongoing cost multiplier at production request volume, not just a one-time design consideration.
- **Security:** since retrieved chunks are inserted directly into the prompt sent to the model, this is exactly the kind of location where a prompt-injection risk could live if retrieved content isn't trusted — content coming from an external or user-editable source should be handled with the same caution as any other untrusted input being placed into a prompt.
- **Deployment and monitoring:** log every chunk-drop event with enough detail to reconstruct why it happened — this is essential for diagnosing whether a bad or incomplete answer was caused by a genuinely relevant chunk being dropped due to budget pressure, versus a retrieval-quality problem, versus something else entirely.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Fixed output reservation vs dynamic reservation:** a single generous fixed reservation for the model's output is simple but wastes budget on requests that only need a short answer, potentially squeezing out genuinely relevant context. A reservation sized to the expected answer type (informed by an earlier classification step, for example) uses the available budget more efficiently, at the cost of added complexity.
- **Dropping from the bottom of the ranking vs some other dropping strategy:** dropping the least relevant already-ranked chunks first, when budget runs out, is the natural choice since it preserves as much of the retrieval and reranking system's quality judgment as possible. Any alternative dropping strategy would need a strong justification to override that ranking.
- **Budgeting as a second, independent safety net rather than the primary limiting mechanism:** reranking already narrows the candidate pool based on relevance quality — this is a quality-driven cutoff. Token budgeting is a capacity-driven cutoff, entirely independent of relevance. If budgeting were the only limiting mechanism, an unusually long system prompt or conversation history could silently squeeze out chunks that reranking correctly identified as highly relevant. The two mechanisms need to compose correctly: reranking picks the best candidates, budgeting then greedily includes as many of those, in rank order, as actually fit.


### 6. Alternatives and When to Use Each

- **Greedy inclusion in rank order (this topic's approach):** the simplest, most defensible strategy — always include the most relevant chunks first, drop from the bottom when budget runs out. The right default for most use cases.
- **Summarizing lower-ranked chunks instead of dropping them entirely:** an alternative worth considering if genuinely useful secondary information is being dropped frequently — trades some fidelity for keeping at least a compressed signal from more content, at the cost of an extra processing step and potential summarization errors.
- **Dynamically adjusting how many chunks to retrieve based on available budget, rather than budgeting after a fixed retrieval count:** shifts some of the capacity constraint earlier in the pipeline, which can be more efficient but requires tighter coupling between the retrieval and prompt construction stages.


### 7. Common Mistakes and Production Failures

- Truncating a chunk mid-sentence to fit the budget instead of dropping it whole — this produces broken, potentially misleading partial information in the model's context.
- Using character-count or word-count as a token proxy without ever calibrating against the real tokenizer, especially risky for a corpus with significant non-English or mixed-language content.
- Forgetting to reserve budget for the model's own output, causing genuinely long, correct answers to be cut off mid-generation.
- Not tagging each chunk with its source at construction time, making later citation attribution difficult or impossible to implement cleanly after the fact.
- Placing the most relevant chunk in the middle of the context block instead of the start or end, silently degrading answer quality via the lost-in-the-middle effect.
- Not logging chunk-drop events, making it impossible to diagnose whether a bad answer was caused by a genuinely relevant chunk that should have been included but wasn't.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why can't retrieved chunks just be concatenated directly into the prompt?
  A: Context windows are finite, and every token spent on chunks is unavailable for the system prompt, conversation history, and the model's own output. Naive concatenation also ignores the lost-in-the-middle effect — chunk order affects how reliably the model uses the content — and provides no mechanism for source attribution needed for citations.

- Q: What is the "lost in the middle" effect and why does it matter for chunk ordering?
  A: Language models are empirically less accurate at using information placed in the middle of a long context compared to the beginning or end, even when all the content technically fits within the context window. This means the most relevant retrieved chunk should be placed at the start or end of the context block, not buried among less relevant chunks.

**Intermediate**

- Q: How would you calculate the actual token budget available for retrieved chunks in a production pipeline?
  A: Start from the model's total context window, subtract the system prompt's token count and any tool-schema tokens (both fixed and measurable), subtract any conversation history tokens for multi-turn cases, and subtract a reserved allocation for the model's output. What remains is the budget available for the context block — greedily fill it with ranked chunks, dropping whole chunks rather than truncating when the budget is exhausted.

- Q: Why is character-count a risky proxy for token-count for a multilingual or code-mixed corpus?
  A: Tokenizers generally handle non-English or code-mixed text less efficiently per character than clean English, meaning a character-based estimate systematically undercounts actual token usage for that content. Near the budget edge, this creates real overflow risk that a purely English-tuned heuristic would never surface in testing.

**Advanced**

- Q: Why is token budgeting a second, independent safety net rather than the primary limiting mechanism for chunk selection?
  A: Reranking already narrows the candidate pool to a final top-k based on relevance quality — a quality-driven cutoff. Token budgeting is a capacity-driven cutoff, entirely independent of relevance. If budgeting were the only limiting mechanism, a request with an unusually long system prompt or conversation history could silently squeeze out chunks that reranking correctly identified as highly relevant, in favor of nothing. The two mechanisms need to compose correctly: reranking picks the best candidates, budgeting then greedily includes as many of those, in rank order, as actually fit, dropping from the bottom of the ranking first if space runs out.

- Q: A teammate proposes dynamically increasing the output token reservation for every request to avoid ever truncating the model's output. What's the risk?
  A: An overly generous fixed reservation, subtracted upfront from the total budget regardless of the actual expected output length, can starve the context block of chunks that were genuinely needed. A better approach ties the reservation to the expected answer type, informed by an upstream classification step, rather than using a single generous constant for every request regardless of its actual needs.

**Scenario-based**

- Q: In production, answers to complex, multi-part questions are frequently incomplete or seem to ignore some of the retrieved context, even though retrieval evaluation metrics show the right chunks are being found. Diagnose.
  A: This is a strong signal to check prompt construction rather than retrieval — specifically the lost-in-the-middle effect and chunk ordering. If retrieval is verified to be correctly finding the right chunks but the model isn't using all of them, check whether the most relevant chunks are being placed in the middle of a large context block rather than at the start or end. Also verify the token budget isn't forcing some genuinely relevant chunks to be silently dropped — cross-reference chunk-drop logs against the specific queries reported as incomplete.

**Follow-up questions to expect:**

- "How would you measure whether your chunk ordering strategy is actually helping, versus just assumed to help?"
- "What would you change about your budgeting logic if the system prompt itself grew significantly larger over time?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Tokenization is model-specific:** different models use different tokenizers, so a token-count estimate calibrated against one model's tokenizer doesn't transfer to another. Any token budgeting logic tied to a specific model choice needs to be re-validated if the underlying model ever changes — model swaps have budget-invalidating side effects, not just quality ones.
- **System prompt token cost compounds across every request:** the system prompt is sent in full on every single request — its token count is a fixed tax on every request's available context budget, making a bloated system prompt a direct, ongoing cost and capacity drain, not a one-time design cost.
- **Prompt caching as a future optimization:** because the system prompt and tool schema are typically identical across many consecutive requests, caching mechanisms exist in many LLM platforms that can reduce the effective cost of resending them every time — worth knowing this exists as an optimization lever once the fixed-cost token tax described above becomes measurable at production volume.

### 10. Quick Revision Summary (for last-minute interview prep)

> Prompt construction turns a ranked chunk list into the actual context block sent to the model. The core discipline is token budgeting: total context window minus system prompt tokens minus reserved output tokens equals the available budget, filled greedily in rank order, dropping whole chunks rather than truncating them when the budget runs out. Token estimation must account for non-English or mixed-language content, where character-count heuristics are unreliable. Chunk ordering matters independently of budget — the most relevant chunk belongs at the start or end of the context block to avoid the lost-in-the-middle effect. Every chunk must be source-tagged at construction time to make later citation attribution possible. Budgeting is a capacity-driven safety net layered on top of reranking's quality-driven cutoff — the two mechanisms are independent and must compose correctly, with budget shortfalls dropping the least relevant already-ranked chunks first.


### Module 1: Token Estimation and Its Language-Dependent Bias

Build a rough character-based token estimator, then demonstrate its systematic bias on English vs Hinglish text.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Token estimation and its language-dependent bias
# ------------------------------------------------------------------

def estimate_tokens(text: str) -> int:
    """Rough approximation: ~4 characters per token for clean English.
    Less reliable for Hinglish/mixed-script text -- always prefer a
    real tokenizer call when precision matters (e.g. near a budget edge)."""
    return len(text) // 4


ENGLISH_TEXT = "What is the penalty for premature withdrawal of a fixed deposit account?"
HINGLISH_TEXT = "Mera FD ka byaj kab milega aur maturity date kya hai iske baare mein bataiye"

# a real tokenizer would generally need MORE tokens per character for
# code-mixed or non-Latin-derived vocabulary than for clean English --
# we simulate this known effect with a simple multiplier, since we do
# not have access to Claude's actual tokenizer offline
SIMULATED_REAL_TOKEN_MULTIPLIER_ENGLISH = 1.0
SIMULATED_REAL_TOKEN_MULTIPLIER_HINGLISH = 1.35  # illustrative, not an exact measured value

english_estimate = estimate_tokens(ENGLISH_TEXT)
hinglish_estimate = estimate_tokens(HINGLISH_TEXT)

simulated_real_english = int(english_estimate * SIMULATED_REAL_TOKEN_MULTIPLIER_ENGLISH)
simulated_real_hinglish = int(hinglish_estimate * SIMULATED_REAL_TOKEN_MULTIPLIER_HINGLISH)

print("=" * 70)
print("TOKEN ESTIMATION: character-based heuristic vs simulated real tokenizer")
print("=" * 70)
print(f"English text ({len(ENGLISH_TEXT)} chars): '{ENGLISH_TEXT[:50]}...'")
print(f"  Character-based estimate: {english_estimate} tokens")
print(f"  Simulated real tokenizer: {simulated_real_english} tokens")
print(f"  Underestimate: {simulated_real_english - english_estimate} tokens "
      f"({(simulated_real_english/english_estimate - 1)*100:.0f}% off)\n")

print(f"Hinglish text ({len(HINGLISH_TEXT)} chars): '{HINGLISH_TEXT[:50]}...'")
print(f"  Character-based estimate: {hinglish_estimate} tokens")
print(f"  Simulated real tokenizer: {simulated_real_hinglish} tokens")
print(f"  Underestimate: {simulated_real_hinglish - hinglish_estimate} tokens "
      f"({(simulated_real_hinglish/hinglish_estimate - 1)*100:.0f}% off)")

print("\nNOTE: the 1.35x multiplier above is illustrative, not a measured")
print("value from Claude's actual tokenizer -- the pedagogical point is")
print("that character-based estimates have DIFFERENT error rates across")
print("languages, and that error is not zero even for English. Near a")
print("hard budget limit, this systematic gap is exactly what causes")
print("silent context overflow in production.")
print("\nModule 1 complete. Run Module 2 next.")


TOKEN ESTIMATION: character-based heuristic vs simulated real tokenizer
English text (72 chars): 'What is the penalty for premature withdrawal of a ...'
  Character-based estimate: 18 tokens
  Simulated real tokenizer: 18 tokens
  Underestimate: 0 tokens (0% off)

Hinglish text (76 chars): 'Mera FD ka byaj kab milega aur maturity date kya h...'
  Character-based estimate: 19 tokens
  Simulated real tokenizer: 25 tokens
  Underestimate: 6 tokens (32% off)

NOTE: the 1.35x multiplier above is illustrative, not a measured
value from Claude's actual tokenizer -- the pedagogical point is
that character-based estimates have DIFFERENT error rates across
languages, and that error is not zero even for English. Near a
hard budget limit, this systematic gap is exactly what causes
silent context overflow in production.

Module 1 complete. Run Module 2 next.


### Module 2: Greedy Chunk Selection Under Budget

Build the actual budgeting function: walk ranked chunks, include what fits, drop whole chunks (never truncate) once the budget runs out.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Greedy chunk selection under a token budget
# ------------------------------------------------------------------

RANKED_CHUNKS = [
    # (relevance_score, {"text": ..., "source": ...}) -- already ranked
    # by the retrieval + rerank pipeline from Chapter 7
    (0.95, {"text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf"}),
    (0.88, {"text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.", "source": "04_FD_Policy_Reference.pdf"}),
    (0.71, {"text": "Step 1 of FD closure: submit a written closure request form to the branch, along with the passbook and identity proof for verification.", "source": "03_FD_SOPs.pdf"}),
    (0.54, {"text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures regardless of the deposit amount or duration selected.", "source": "02_FD_Product_Guide.pdf"}),
    (0.31, {"text": "Nomination facility is available for all Fixed Deposit account holders and can be updated at any branch at any time free of charge.", "source": "02_FD_Product_Guide.pdf"}),
]


def build_context_block(ranked_chunks: list, token_budget: int) -> tuple:
    """Greedily includes chunks in rank order until the budget is exhausted.
    Never truncates a chunk mid-sentence -- drops the whole chunk instead.
    Returns (context_block_string, list_of_dropped_chunks)."""
    included = []
    dropped = []
    used_tokens = 0
    for score, doc in ranked_chunks:
        chunk_text = f"[Source: {doc['source']}]\n{doc['text']}\n"
        chunk_tokens = estimate_tokens(chunk_text)
        if used_tokens + chunk_tokens > token_budget:
            dropped.append((score, doc))
            continue  # skip this one, but keep checking smaller chunks further down
        included.append(chunk_text)
        used_tokens += chunk_tokens
    return "\n---\n".join(included), dropped, used_tokens


print("=" * 70)
print("GREEDY CHUNK SELECTION AT DIFFERENT BUDGET LEVELS")
print("=" * 70)

for budget in [200, 80, 40]:
    context_block, dropped, used = build_context_block(RANKED_CHUNKS, token_budget=budget)
    included_count = len(RANKED_CHUNKS) - len(dropped)
    print(f"\nBudget = {budget} tokens:")
    print(f"  Included {included_count}/{len(RANKED_CHUNKS)} chunks, used {used} tokens")
    if dropped:
        dropped_sources = [d["source"] for _, d in dropped]
        print(f"  DROPPED (logged): {dropped_sources}")
    else:
        print("  No chunks dropped.")

print("\nAt budget=40, notice which chunks survive -- the greedy algorithm")
print("always protects the HIGHEST-ranked chunks first, dropping from the")
print("bottom of the ranking, never truncating a surviving chunk mid-sentence.")
print("\nModule 2 complete. Run Module 3 next.")


GREEDY CHUNK SELECTION AT DIFFERENT BUDGET LEVELS

Budget = 200 tokens:
  Included 5/5 chunks, used 187 tokens
  No chunks dropped.

Budget = 80 tokens:
  Included 2/5 chunks, used 62 tokens
  DROPPED (logged): ['03_FD_SOPs.pdf', '02_FD_Product_Guide.pdf', '02_FD_Product_Guide.pdf']

Budget = 40 tokens:
  Included 1/5 chunks, used 27 tokens
  DROPPED (logged): ['04_FD_Policy_Reference.pdf', '03_FD_SOPs.pdf', '02_FD_Product_Guide.pdf', '02_FD_Product_Guide.pdf']

At budget=40, notice which chunks survive -- the greedy algorithm
always protects the HIGHEST-ranked chunks first, dropping from the
bottom of the ranking, never truncating a surviving chunk mid-sentence.

Module 2 complete. Run Module 3 next.


### Module 3: Full Budget Calculation and Prompt Assembly

Puts it all together: compute the real available budget after accounting for system prompt and reserved output, then assemble the final prompt.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Full budget calculation and final prompt assembly
# ------------------------------------------------------------------

def build_rag_prompt(query: str, ranked_chunks: list, system_prompt_tokens: int,
                      context_window: int = 200_000, reserved_output: int = 1000) -> dict:
    """Computes the real available budget, builds the context block,
    and assembles the final user message sent to the model."""
    available = context_window - system_prompt_tokens - reserved_output
    context_block, dropped, used = build_context_block(ranked_chunks, token_budget=available)
    user_message = f"Context:\n{context_block}\n\nCustomer question: {query}"
    return {
        "role": "user",
        "content": user_message,
        "_debug_available_budget": available,
        "_debug_tokens_used": used,
        "_debug_dropped_chunks": [d["source"] for _, d in dropped],
    }


QUERY = "What is the penalty if I want to close my FD early, and what documents do I need?"
SYSTEM_PROMPT_TOKENS = 850   # measured from the actual system prompt in a real pipeline

result = build_rag_prompt(
    query=QUERY,
    ranked_chunks=RANKED_CHUNKS,
    system_prompt_tokens=SYSTEM_PROMPT_TOKENS,
    context_window=200_000,
    reserved_output=1000,
)

available_budget = result["_debug_available_budget"]
tokens_used = result["_debug_tokens_used"]
dropped_chunks = result["_debug_dropped_chunks"]

print("=" * 70)
print("FULL PROMPT ASSEMBLY")
print("=" * 70)
print(f"Context window:        200,000 tokens")
print(f"System prompt tokens:  {SYSTEM_PROMPT_TOKENS}")
print(f"Reserved output:       1,000 tokens")
print(f"Available for chunks:  {available_budget:,} tokens")
print(f"Actually used:         {tokens_used} tokens")
print(f"Dropped chunks:        {dropped_chunks or 'none'}")

print(f"\nFinal message content (truncated for display):")
print(result["content"][:400] + "...")

print("\nWith a 200K context window, this small demo corpus fits entirely")
print("with enormous headroom to spare -- the REAL constraint in production")
print("shows up with much larger candidate pools, longer conversation")
print("history (Topic 6), or smaller-context models. The budgeting logic")
print("itself does not change; only the numbers involved do.")

print("\nModule 3 complete. All Chapter 8 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 8 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  available_budget = context_window - system_prompt_tokens - reserved_output

  Greedily include ranked chunks until budget is exhausted.
  NEVER truncate a chunk mid-sentence -- drop it whole instead, and LOG
  every drop for debugging.

  Character-count token estimates are systematically less accurate for
  non-English / code-mixed text -- calibrate against the real tokenizer
  near budget-critical paths.

  Budgeting is a CAPACITY-driven safety net, separate from reranking's
  QUALITY-driven cutoff -- both must compose correctly.

  Source-tag every chunk AT CONSTRUCTION TIME -- this is required for
  citation attribution (Topic 2) and cannot be cleanly added afterward.
""")


FULL PROMPT ASSEMBLY
Context window:        200,000 tokens
System prompt tokens:  850
Reserved output:       1,000 tokens
Available for chunks:  198,150 tokens
Actually used:         187 tokens
Dropped chunks:        none

Final message content (truncated for display):
Context:
[Source: 01_FD_FAQ.pdf]
Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.

---
[Source: 04_FD_Policy_Reference.pdf]
Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.

---
[Source: 03_FD_SOPs.pdf]
Step 1 of FD closure: submit a written closure request form to the branch, along with the passbook and i...

With a 200K context window, this small demo corpus fits entirely
with enormous headroom to spare -- the REAL constraint in production
shows up with much larger candidate pools, longer conversation
history (Topic 6), or smaller-context models. The budgeting logic
itself does not change; only the numbers involved do.

Module 